# Modelado de Fatiga con xLSTM (Extended Long Short-Term Memory) - sLSTM Personalizada

Este notebook contiene la explicación teórica, la revisión de literatura científica, la arquitectura detallada, y la implementación paso a paso de una red **sLSTM (Stabilized LSTM - parte de la familia xLSTM) implementada manualmente en PyTorch** para predecir los niveles continuos de fatiga física y mental del dataset **FatigueSet**.

---

## 1. Fundamentos Teóricos y Literatura de Referencia

La arquitectura **xLSTM (Extended LSTM)** fue propuesta por Beck et al. en 2024 para actualizar la LSTM clásica y permitirle escalar de forma competitiva frente a los Transformers. xLSTM introduce dos modificaciones principales: puertas exponenciales y estructuras de almacenamiento mejoradas (sLSTM con memoria escalar estabilizada y mLSTM con memoria matricial paralela).

En este notebook nos enfocamos en **sLSTM (Stabilized LSTM)**, la cual sustituye la activación sigmoide tradicional de las compuertas de olvido y entrada por activaciones exponenciales, permitiendo un direccionamiento de memoria mucho más dinámico. Para evitar el desbordamiento numérico (overflow) por los términos exponenciales, se introduce un término normalizador ($n_t$) y un estado de seguimiento del máximo en escala logarítmica ($m_t$).

### Formulación Matemática

Para un paso de tiempo $t$, dada la entrada $x_t$, el estado oculto previo $h_{t-1}$, el estado de celda previo $c_{t-1}$, el normalizador previo $n_{t-1}$ y el log-máximo previo $m_{t-1}$, calculamos:

1. **Pre-activaciones lineales de las Compuertas:**
   $$\tilde{f}_t = W_f x_t + U_f h_{t-1} + b_f$$
   $$\tilde{i}_t = W_i x_t + U_i h_{t-1} + b_i$$
   $$\tilde{c}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)$$
   $$\tilde{o}_t = W_o x_t + U_o h_{t-1} + b_o$$

2. **Actualización del Log-Normalizador de Estabilización ($m_t$):**
   $$m_t = \max(m_{t-1} + \tilde{f}_t, \tilde{i}_t)$$

3. **Compuertas Exponenciales Estabilizadas:**
   $$f_t^s = \exp(m_{t-1} + \tilde{f}_t - m_t)$$
   $$i_t^s = \exp(\tilde{i}_t - m_t)$$

4. **Actualización del Estado de Celda ($c_t$) y Normalizador ($n_t$):**
   $$c_t = f_t^s c_{t-1} + i_t^s \tilde{c}_t$$
   $$n_t = f_t^s n_{t-1} + i_t^s$$

5. **Normalización del Estado de la Celda y Salida Oculta ($h_t$):**
   $$\hat{c}_t = \frac{c_t}{n_t}$$
   $$o_t = \sigma(\tilde{o}_t)$$
   $$h_t = o_t \odot \tanh(\hat{c}_t)$$

---

### Diagrama de Flujo de la Celda (Mermaid)

```mermaid
graph TD
    subgraph "Celda sLSTM (Paso t)"
        xt["Entrada actual: x_t"]
        h_prev["Estado oculto anterior: h_{t-1}"]
        c_prev["Estado de celda anterior: c_{t-1}"]
        n_prev["Normalizador anterior: n_{t-1}"]
        m_prev["Log-máximo anterior: m_{t-1}"]

        f_tilde["Forget Pre-act: W_f x_t + U_f h_{t-1} + b_f"]
        i_tilde["Input Pre-act: W_i x_t + U_i h_{t-1} + b_i"]
        c_tilde["Candidato Celda: c̃_t = tanh(W_c x_t + U_c h_{t-1} + b_c)"]
        o_tilde["Output Pre-act: W_o x_t + U_o h_{t-1} + b_o"]

        m_t["Max Update: m_t = max(m_{prev} + f_tilde, i_tilde)"]
        f_s["Stabilized Forget: f_s = exp(m_{prev} + f_tilde - m_t)"]
        i_s["Stabilized Input: i_s = exp(i_tilde - m_t)"]

        c_update["Celda Update: c_t = f_s c_{prev} + i_s c̃_t"]
        n_update["Normalizador Update: n_t = f_s n_{prev} + i_s"]
        c_hat["Normalized Cell: ĉ_t = c_t / n_t"]

        o_t["Output Gate: o_t = σ(o_tilde)"]
        h_update["Hidden Update: h_t = o_t ⊙ tanh(ĉ_t)"]

        xt --> f_tilde
        xt --> i_tilde
        xt --> c_tilde
        xt --> o_tilde

        h_prev --> f_tilde
        h_prev --> i_tilde
        h_prev --> c_tilde
        h_prev --> o_tilde

        m_prev --> m_t
        f_tilde --> m_t
        i_tilde --> m_t

        m_prev --> f_s
        f_tilde --> f_s
        m_t --> f_s

        i_tilde --> i_s
        m_t --> i_s

        c_prev --> c_update
        f_s --> c_update
        i_s --> c_update
        c_tilde --> c_update

        n_prev --> n_update
        f_s --> n_update
        i_s --> n_update

        c_update --> c_hat
        n_update --> c_hat

        c_hat --> h_update
        o_tilde --> o_t
        o_t --> h_update
    end
```

---

### Citas Bibliográficas Científicas

* **Beck, M., Pöppel, K., Spanring, T., Auer, A., Prudnikova, O., Kopp, M., Klambauer, G., Brandstetter, J., & Hochreiter, S. (2024).** *xLSTM: Extended Long Short-Term Memory*. arXiv preprint arXiv:2405.04517. [Enlace al Paper](https://arxiv.org/abs/2405.04517)

In [1]:
# SETUP e IMPORTACIONES
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Añadir fatigueset-lib al sys.path
lib_path = str(Path.cwd().parent / "fatigueset-lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from fatigueset import FatigueSetPipeline
from fatigueset.models import CustomxLSTMRegressor, FatigueSequenceDataset
from fatigueset.models.rnn import _prepare_target_table, _merge_raw_streams, _build_sequences

print("[OK] Imports completados y path configurado.")
print(f"Dispositivo actual: {'cuda' if torch.cuda.is_available() else 'cpu'}")

[OK] Imports completados y path configurado.
Dispositivo actual: cuda


## 2. Configuración del Pipeline y Construcción de Secuencias

In [2]:
# Configuración del dataset y pipeline
dataset_path = str(Path.cwd().parent / "fatigueset")
pipeline = FatigueSetPipeline(dataset_path=dataset_path, umbral_nulos=5.0)

print("Cargando dataset...")
raw = pipeline.cargar_dataset(verbose=False)

print("Preparando targets del dataframe ML...")
df_ml = pipeline.construir_dataset_ml(raw)
df_targets = _prepare_target_table(df_ml)

print("Combinando streams fisiológicos crudos...")
df_raw = _merge_raw_streams(raw)

# Parámetros de ventanas de secuencia temporal
seq_len = 128
step = 32

print(f"Construyendo secuencias de tamaño={seq_len} y paso={step}...")
X_arr, y_arr, groups, feature_cols = _build_sequences(
    df_raw=df_raw,
    df_targets=df_targets,
    seq_len=seq_len,
    step=step
)

print(f"[OK] Dimensiones de tensores construidos:")
print(f"  - X: {X_arr.shape} (Número de ventanas x seq_len x features)")
print(f"  - y: {y_arr.shape} (Número de ventanas x 2 targets)")

Cargando dataset...


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


Preparando targets del dataframe ML...


Combinando streams fisiológicos crudos...


Construyendo secuencias de tamaño=128 y paso=32...
[OK] Dimensiones de tensores construidos:
  - X: (1306, 128, 23) (Número de ventanas x seq_len x features)
  - y: (1306, 2) (Número de ventanas x 2 targets)


## 3. División de Datos por Participante (Group Split) y DataLoaders

Para garantizar la rigurosidad científica y evitar la fuga de información (data leakage), separamos el participante '01' para validación.

In [3]:
train_idx = np.where(groups != '01')[0]
val_idx = np.where(groups == '01')[0]

X_train, y_train = X_arr[train_idx], y_arr[train_idx]
X_val, y_val = X_arr[val_idx], y_arr[val_idx]

train_dataset = FatigueSequenceDataset(X_train, y_train)
val_dataset = FatigueSequenceDataset(X_val, y_val)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Train samples: 1184
Validation samples: 122


## 4. Inicialización del Regresor Custom sLSTM (xLSTM)

Instanciamos nuestro regresor `CustomxLSTMRegressor`.

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

input_size = len(feature_cols)
hidden_size = 64
num_layers = 2
dropout = 0.2

model = CustomxLSTMRegressor(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    output_size=2
).to(device)

print(model)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Número de parámetros entrenables: {num_params:,}")

CustomxLSTMRegressor(
  (xlstm): CustomxLSTM(
    (layers): ModuleList(
      (0-1): 2 x CustomxLSTMCell()
    )
    (dropout_layer): Dropout(p=0.2, inplace=False)
  )
  (fc): Linear(in_features=64, out_features=2, bias=True)
)
Número de parámetros entrenables: 55,682


## 5. Entrenamiento Corto de Validación (Sanity Check)

Ejecutamos 5 épocas sobre los datos para validar que las puertas exponenciales y la normalización estabilizada en PyTorch sean correctas.

In [5]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
print("Iniciando mini-entrenamiento de sLSTM...")

for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        
        # Gradient clipping para evitar inestabilidad recurrente por exponenciales
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
    
    print(f"Epoch {epoch}/{epochs} - Train Loss: {avg_train_loss:.6f} - Val Loss: {avg_val_loss:.6f}")

print("[OK] Entrenamiento corto finalizado correctamente.")

Iniciando mini-entrenamiento de sLSTM...


Epoch 1/5 - Train Loss: 1256.723227 - Val Loss: 1162.538544


Epoch 2/5 - Train Loss: 1080.162139 - Val Loss: 1001.727310


Epoch 3/5 - Train Loss: 965.830868 - Val Loss: 879.954117


Epoch 4/5 - Train Loss: 875.119814 - Val Loss: 772.554413


Epoch 5/5 - Train Loss: 794.040460 - Val Loss: 675.675568
[OK] Entrenamiento corto finalizado correctamente.


## 6. Serialización del Modelo

Guardamos los pesos del modelo sLSTM en la carpeta centralizada `/models/deep_learning/`.

In [6]:
output_dir = Path.cwd().parent / "models" / "deep_learning"
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "xlstm_fatigue_notebook.pt"
torch.save(model.state_dict(), model_path)

print(f"[OK] Modelo sLSTM guardado exitosamente en: {model_path}")

[OK] Modelo sLSTM guardado exitosamente en: C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\models\deep_learning\xlstm_fatigue_notebook.pt
